# Data Pipelines & Feature Engineering Cheatsheet

> Building reliable data pipelines, feature stores, and data quality frameworks for ML.

---
## Apache Airflow

### Setup
```bash
pip install apache-airflow
airflow db init
airflow users create --username admin --firstname Admin --lastname User \
  --role Admin --email admin@example.com --password admin
airflow webserver --port 8080 &
airflow scheduler &
```

In [ ]:
# Airflow ML Pipeline DAG (reference code)
from datetime import datetime, timedelta
# from airflow import DAG
# from airflow.operators.python import PythonOperator
# from airflow.operators.bash import BashOperator

default_args = {
    "owner": "ml-team",
    "depends_on_past": False,
    "start_date": datetime(2024, 1, 1),
    "retries": 2,
    "retry_delay": timedelta(minutes=5),
    "email_on_failure": True,
}

# dag = DAG(
#     "ml_training_pipeline",
#     default_args=default_args,
#     description="End-to-end ML training pipeline",
#     schedule_interval="@daily",
#     catchup=False,
# )

def extract_data(**kwargs):
    print("Extracting data from source...")
    return {"rows": 10000, "source": "database"}

def validate_data(**kwargs):
    ti = kwargs["ti"]
    data_info = ti.xcom_pull(task_ids="extract")
    print(f"Validating {data_info['rows']} rows...")
    return True

def train_model(**kwargs):
    print("Training model...")
    return {"accuracy": 0.95, "model_path": "models/v1.pkl"}

def evaluate_model(**kwargs):
    ti = kwargs["ti"]
    train_result = ti.xcom_pull(task_ids="train")
    print(f"Model accuracy: {train_result['accuracy']}")

# Pipeline: extract >> validate >> [feature_eng, train] >> evaluate >> deploy
print("Airflow ML DAG pattern defined")
print(f"Default args: {default_args}")

### Airflow CLI

```bash
# DAG management
airflow dags list                      # List all DAGs
airflow dags trigger ml_training_pipeline  # Trigger run
airflow dags pause ml_training_pipeline    # Pause
airflow dags unpause ml_training_pipeline  # Unpause

# Task management
airflow tasks list ml_training_pipeline      # List tasks
airflow tasks test ml_training_pipeline train 2024-01-01  # Test task
airflow tasks run ml_training_pipeline train 2024-01-01   # Run task

# Monitoring
airflow dags state ml_training_pipeline 2024-01-01        # Check state
airflow tasks states-for-dag-run ml_training_pipeline 2024-01-01  # Task states
```

---
## Prefect

In [ ]:
# Prefect ML Pipeline (reference code)
# from prefect import flow, task
# from prefect.tasks import task_input_hash
# from datetime import timedelta

# @task(retries=3, cache_key_fn=task_input_hash, cache_expiration=timedelta(hours=1))
# def extract_data(source: str):
#     print(f"Extracting from {source}...")
#     return {"rows": 10000}

# @task
# def transform_data(raw_data: dict):
#     print(f"Transforming {raw_data['rows']} rows...")
#     return {"features": 50, "rows": raw_data["rows"]}

# @task
# def train(data: dict):
#     print(f"Training on {data['rows']} rows with {data['features']} features")
#     return {"accuracy": 0.95}

# @flow(name="ml-training-pipeline")
# def ml_pipeline(data_source: str = "s3://bucket/data/"):
#     raw = extract_data(data_source)
#     features = transform_data(raw)
#     result = train(features)
#     return result

# Run: ml_pipeline()
# Deploy: prefect deploy ml_pipeline --name daily-training --cron "0 2 * * *"
print("Prefect pipeline pattern: @task for steps, @flow for orchestration")

---
## Great Expectations (Data Quality)

In [ ]:
# Great Expectations Data Validation (reference code)
# import great_expectations as gx

# context = gx.get_context()

# # Connect to data
# datasource = context.sources.add_pandas("my_datasource")
# data_asset = datasource.add_csv_asset("training_data", filepath_or_buffer="data/train.csv")

# # Define expectations
# suite = context.add_expectation_suite("training_data_quality")

# validator = context.get_validator(
#     batch_request=data_asset.build_batch_request(),
#     expectation_suite_name="training_data_quality"
# )

# validator.expect_column_values_to_not_be_null("target")
# validator.expect_column_values_to_be_between("age", min_value=0, max_value=120)
# validator.expect_column_values_to_be_in_set("category", ["A", "B", "C"])
# validator.expect_column_mean_to_be_between("amount", min_value=10, max_value=1000)
# validator.expect_table_row_count_to_be_between(min_value=1000, max_value=1000000)

# # Validate
# results = validator.validate()
# print(f"Validation passed: {results.success}")

print("Great Expectations: define data quality rules, validate before training")

---
## Streaming Data (Kafka)

In [ ]:
# Kafka ML Feature Streaming (reference code)
from kafka import KafkaConsumer, KafkaProducer
import json

# Producer: send features
# producer = KafkaProducer(
#     bootstrap_servers=["localhost:9092"],
#     value_serializer=lambda v: json.dumps(v).encode("utf-8")
# )

# producer.send("ml-features", {
#     "user_id": "user123",
#     "features": {"transaction_amount": 500, "location": "NYC"},
#     "timestamp": "2024-01-01T12:00:00Z"
# })

# Consumer: process features for real-time inference
# consumer = KafkaConsumer(
#     "ml-features",
#     bootstrap_servers=["localhost:9092"],
#     value_deserializer=lambda m: json.loads(m.decode("utf-8")),
#     auto_offset_reset="latest",
#     group_id="ml-inference-group"
# )

# for message in consumer:
#     features = message.value
#     prediction = model.predict(features)
#     producer.send("ml-predictions", {"prediction": prediction, **features})

print("Kafka pattern: produce features → consume → predict → produce results")

---
## Data Lake Architecture (Medallion)

```
┌─────────────┐     ┌──────────────┐     ┌──────────────┐
│   BRONZE     │────▶│   SILVER      │────▶│   GOLD        │
│  (Raw Data)  │     │  (Cleaned)    │     │  (Features)   │
│              │     │              │     │              │
│ - Raw files  │     │ - Deduped    │     │ - ML-ready   │
│ - API dumps  │     │ - Validated  │     │ - Aggregated │
│ - Logs       │     │ - Typed      │     │ - Joined     │
└─────────────┘     └──────────────┘     └──────────────┘
```

In [ ]:
# Delta Lake Pattern (reference code)
# from delta import DeltaTable
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName("ML-Pipeline") \
#     .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
#     .getOrCreate()

# Bronze: read raw data
# raw_df = spark.read.json("s3://data-lake/bronze/events/")
# raw_df.write.format("delta").mode("append").save("s3://data-lake/bronze/events_delta/")

# Silver: clean and transform
# silver_df = raw_df.dropDuplicates(["event_id"]).filter("amount > 0")
# silver_df.write.format("delta").mode("overwrite").save("s3://data-lake/silver/transactions/")

# Gold: create ML features
# gold_df = silver_df.groupBy("user_id").agg(
#     F.avg("amount").alias("avg_amount"),
#     F.count("*").alias("tx_count"),
#     F.stddev("amount").alias("amount_stddev")
# )
# gold_df.write.format("delta").mode("overwrite").save("s3://data-lake/gold/user_features/")

# Time travel: query previous version
# df_v0 = spark.read.format("delta").option("versionAsOf", 0).load("s3://data-lake/gold/user_features/")

print("Medallion architecture: Bronze (raw) → Silver (clean) → Gold (features)")

---
## Cloud Data Services

### AWS
```bash
# S3 data operations
aws s3 cp data/ s3://my-ml-bucket/data/ --recursive
aws s3 sync s3://my-ml-bucket/data/ ./data/

# Glue crawler (auto-detect schema)
aws glue start-crawler --name my-data-crawler

# Athena query (SQL on S3)
aws athena start-query-execution \
  --query-string "SELECT * FROM my_table LIMIT 10" \
  --result-configuration '{"OutputLocation": "s3://my-bucket/results/"}'
```

### Azure
```bash
# Blob storage
az storage blob upload-batch -d my-container -s ./data/ --account-name mystorageaccount

# Synapse SQL
az synapse sql-script create --workspace my-synapse --name query1 \
  --sql-content "SELECT * FROM training_data LIMIT 10"

# Data Factory pipeline trigger
az datafactory pipeline create-run --factory-name my-factory --name ml-data-pipeline
```

### GCP
```bash
# GCS operations
gsutil cp -r data/ gs://my-ml-bucket/data/
gsutil rsync -r gs://my-ml-bucket/data/ ./data/

# BigQuery
bq query --use_legacy_sql=false 'SELECT * FROM my_dataset.training_data LIMIT 10'

# Dataflow (Apache Beam)
python pipeline.py --runner DataflowRunner --project my-project --region us-central1
```

## Pipeline Comparison

| Tool | Best For | Complexity | Streaming |
|------|----------|------------|----------|
| **Airflow** | Complex DAGs, batch | Medium | No |
| **Prefect** | Modern ML pipelines | Low | Limited |
| **Kafka** | Real-time streaming | High | Yes |
| **Spark** | Large-scale batch | High | Yes (Structured) |
| **dbt** | SQL transformations | Low | No |

## Interview Scenarios

**Q: How would you design a data pipeline for an ML system?**
> Use medallion architecture (bronze/silver/gold). Bronze ingests raw data, silver cleans and validates (Great Expectations), gold creates ML features. Orchestrate with Airflow/Prefect. Store in Delta Lake for versioning. Add data quality gates between layers. Monitor with alerts on schema changes, volume anomalies, and freshness SLAs.

**Q: How do you handle data quality in ML pipelines?**
> Implement validation at every stage: (1) schema validation on ingestion, (2) statistical tests in silver layer (Great Expectations), (3) feature distribution checks before training, (4) data drift monitoring in production (Evidently). Set up alerts and circuit breakers to prevent bad data from reaching models.

**Q: Batch vs. streaming — when to use each for ML?**
> Batch for: model training, feature engineering on historical data, periodic predictions. Streaming for: real-time inference, online feature computation, fraud detection, recommendations. Often use a lambda architecture: batch for training + streaming for serving.